In [1]:
!sudo apt install unrar
!unrar x second_batch_fixed.rar

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
unrar is already the newest version (1:6.1.5-1ubuntu0.1).
0 upgraded, 0 newly installed, 0 to remove and 35 not upgraded.

UNRAR 6.11 beta 1 freeware      Copyright (c) 1993-2022 Alexander Roshal


Extracting from second_batch_fixed.rar

Creating    second_batch                                              OK
Creating    second_batch/commercial_consumption                       OK
Extracting  second_batch/commercial_consumption/consumption_2024-05-31_to_2024-06-10.csv       0%  OK 
Extracting  second_batch/commercial_consumption/consumption_2024-06-11_to_2024-06-21.csv       0%  OK 
Extracting  second_batch/commercial_consumption/consumption_2024-06-22_to_2024-07-02.csv       1%  OK 
Extracting  second_batch/commercial_consumption/consumption_2024-07-03_to_2024-07-13.csv       2%  OK 
Extracting  second_batch/commercial_consumption/consumption_2024-07-14_

In [2]:
import pandas as pd
import numpy as np
import os

### Industrial data

Keeping date and target column name consistent across all files

In [3]:
ind_data_dir = '/content/second_batch/industrial/'

# site1, site3 (have similar format)

for i in [1, 3]:
    site_id = 'site'+str(i)
    df = pd.read_csv(ind_data_dir+site_id+'.csv')
    df.rename(columns={'date':'time', '[ kWh ]':'consumption'}, inplace=True)
    df['time'] = pd.to_datetime(df['time'], format='%Y-%m-%d %H:%M:%S')
    df['site_id'] = site_id
    df.to_csv(ind_data_dir+site_id+'.csv', index=False)

In [4]:
# site2 and site9 (have similar format)

for i in [2, 9]:
    site_id = 'site'+str(i)
    df = pd.read_csv(ind_data_dir+site_id+'.csv')
    df.rename(columns={'date':'time', '[ kWh ]':'consumption'}, inplace=True)
    df['time'] = pd.to_datetime(df['time'], format='%Y/%m/%d, %H:%M')
    df['site_id'] = site_id
    df.to_csv(ind_data_dir+site_id+'.csv', index=False)

In [5]:
# site4, site8

for i in [4, 8]:
    site_id = 'site'+str(i)
    df = pd.read_csv(ind_data_dir+site_id+'.csv')
    df.rename(columns={'date':'time', 'avg_power':'consumption'}, inplace=True)
    df['time'] = pd.to_datetime(df['time'], format='%Y/%m/%d, %H:%M')
    df['site_id'] = 'site4'
    df.to_csv(ind_data_dir+site_id+'.csv', index=False)

In [6]:
# site5 (choosing only one among 5, 6, 7)

site5 = pd.read_csv(os.path.join(ind_data_dir, 'site5.csv'))
site5.rename(columns={'date':'time', 'P+ total':'consumption'}, inplace=True)
site5['time'] = pd.to_datetime(df['time'], format='%Y/%m/%d, %H:%M')
site5['site_id'] = 'site5'
site5.to_csv(os.path.join(ind_data_dir, 'site5.csv'), index=False)

### Commercial data

In [7]:
from re import sub

def read_sub_files(path):

    import os

    sub_files = [f for f in os.listdir(path) if os.path.isfile(os.path.join(path, f))]
    # print(sub_files)

    df_list = []
    for file in sub_files:
        df = pd.read_csv(os.path.join(path, file))
        df['time'] = pd.to_datetime(df['ts'], unit='s')
        df = df.drop(columns=['datum', 'ts', 'granularity', 'pin'])
        df = df.rename(columns={'spotreba [kWh]': 'consumption'})
        df_list.append(df)

    sub_concat = pd.concat(df_list, ignore_index=True)
    return sub_concat

In [8]:
# root dir files
commercial_dir = '/content/second_batch/commercial_consumption'

sub2 = read_sub_files(commercial_dir)
sub2.to_csv(os.path.join(commercial_dir, 'sub2.csv'), index=False)

In [16]:
# other sites
for site in ['sub1', 'sub3']:
    df = read_sub_files(os.path.join(commercial_dir, site))
    df.to_csv(os.path.join(commercial_dir, site+'.csv'), index=False)